# 06 - Dashboard and Summary
## Greater London House Price Analysis
Este notebook documenta el dashboard final de las consultas de negocio realizadas, mostrando las más relevantes y consolidando los findings en un formato interactivo y visual.  

### Objetivo
Viaualizar los findings encontrados en la notebook `05_sql_analisis` de manera intuitiva y visual, permitiendo una interacción mas sencilla con los gráficos creados.

### Tabla de entrada
`workspace.default.london_gold` — 2,384,979 filas · 22 columnas

### Pipeline
```
Bronze (raw) → Silver (limpia) → Silver enriquecida (nulos tratados) --> Transformaciones (nuevas variables/KPI´s) --> Capa Gold 
--> SQL Queries --> Valor/Conocimiento disponible para negocio --> Dashboard --> Consolidación de resultados
```

## 1. Visualización de los datos de la capa gold

Se carga el dataset y visualizan las primeras cinco líneas de esta capa gold, para confirmar sus dimensiones y variables. Posteriormente se procede a generar las consultas mas relevantes encontradas.

In [0]:
sql = spark.table("workspace.default.london_gold")
display(sql.limit(5))

##2. Carga de consultas

A continuacion, se cargan las consultas con los findings mas relevantes encontrados. El objetivo, es construir posteriormente un dashboard con los resultados de estas queries en formato grafico para ayudar al entendimiento de negocio. Las consultas seleccionadas son: 

- Evolución del mercado (Tendencia anual)
- Segmentación por Tipo de Propiedad (Últimos 5 años)
- Top 10 Distritos más Caros (Media Histórica)
- Brecha de Precio: Obra Nueva vs. Segunda Mano

In [0]:
%sql
SELECT 
    year, 
    avg_price_annual AS precio_medio_londres,
    COUNT(transaction_id) AS volumen_ventas
FROM workspace.default.london_gold
GROUP BY year, avg_price_annual
ORDER BY year DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    year, 
    property_type_desc, 
    ROUND(AVG(price), 0) AS precio_medio
FROM workspace.default.london_gold
WHERE year >= 2020
GROUP BY year, property_type_desc
ORDER BY year DESC, precio_medio DESC;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    district, 
    ROUND(AVG(price), 0) AS precio_medio_historico,
    COUNT(transaction_id) AS total_transacciones
FROM workspace.default.london_gold
GROUP BY district
ORDER BY precio_medio_historico DESC
LIMIT 10;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    year, 
    property_status, 
    ROUND(AVG(price), 0) AS precio_medio
FROM workspace.default.london_gold
GROUP BY year, property_status
ORDER BY year DESC, property_status;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    transaction_id,
    date_of_transfer,
    district,
    postcode,
    price AS precio_venta,
    avg_price_by_district AS media_distrito_ese_anio,
    price_vs_district_pct AS porcentaje_desviacion
FROM workspace.default.london_gold
WHERE year = 2024 
  AND price_vs_district_pct < -25  -- Un 25% más barato que la media del barrio
ORDER BY price_vs_district_pct ASC
LIMIT 20;

Databricks visualization. Run in Databricks to view.

## 3. Elaboración del cuadro de mando

A continuación, se procede a juntar los outputs en un solo cuadro de mando listo para tomar decisiones de negocio. Se ejecuta usando la funcion `notebook dashboard` de Databricks, donde se van a consolidar todas las visualizaciones creadas en un dashboard interactivo.

- Se accede de la siguiente manera: `View --> Views --> UK House Prices - Dashboard`
- Una vez dentro, para visualizar a pantalla completa --> `Focus`